# AI Engineering Platform ? From-Scratch Language Model

??? ?????? ????? ???? ????? ????? Transformer ?? ????? ???????? ??? ????? ????? ?????? ?????? ?? ??????? ?????? ????? ???? ??????? ??????.

**??? ???????:** ?? ????? Colab ???? `Runtime ? Change runtime type ? T4 GPU` ?? ??? ??????. ???? ????? ???????? ??? CPU ?????? ???? ?? ???? ????.

**???:** ?????? ???????? ???????? ????? ?????? ?? ????? ?? ??????? ???? ??? ???? ??????? ????? ?? ???? ?????.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/content/ai-engineering-platform')
if not REPO.exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/F13342/ai-engineering-platform.git',
        str(REPO),
    ], check=True)

os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))

try:
    import torch
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch'], check=True)
    import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Project:', REPO)


## 1. ????? ?????????? ???????

????? ??? ?????? ???????? ???????? ??????? ?????????? ????????? ???????? ?????? ?????? ???????. ??? ??? ??????? ????? ??????? ????? ??? ?????? ??????? ??????.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pytest'], check=True)
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_language_model.py',
    'tests/test_language_model_tokenizer.py',
    'tests/test_language_model_training.py',
], check=True)


## 2. ????? ?????? ?? ?????

?????? ???? ???????? ????????? ?????? ?????? ??? ?? ??????? ???. ?? ?????? ??? ???????? ?????? ???? ??????? ?? ??????? ??????? ?????? ???????.

In [ ]:
from ai_platform.machine_learning.language_model.training import (
    LanguageModelTrainingConfig,
    train_language_model,
    load_language_model,
    generate_text,
)

corpus_text = (
    'AI engineering platform trains its own language model from random initialization. '
    'A tokenizer converts characters into token IDs. A transformer learns to predict '
    'the next character from previous characters. The training loop measures loss, '
    'updates model parameters, and saves checkpoints. '
) * 120

corpus_path = Path('/content/model_lab_smoke_corpus.txt')
corpus_path.write_text(corpus_text, encoding='utf-8')

output_dir = Path('/content/model_lab_smoke_artifacts')
config = LanguageModelTrainingConfig(
    block_size=32,
    n_layers=2,
    n_heads=2,
    n_embd=64,
    dropout=0.1,
    batch_size=16,
    max_steps=30,
    learning_rate=3e-4,
    eval_interval=10,
    eval_batches=3,
    train_fraction=0.9,
    seed=42,
    device='auto',
)

result = train_language_model(corpus_path, output_dir, config)
print('\nTraining result:')
print(result)


## 3. ????? ????? ??????? ?????? ??

????? ???? checkpoint ??????? ?? ??????? ??????? ?????? ?? ??????? ???????? ??? ?????? ???? ???????.

In [ ]:
checkpoint_path = output_dir / 'best.pt'
model, tokenizer, checkpoint = load_language_model(checkpoint_path, device='auto')

print('Checkpoint step:', checkpoint['step'])
print('Vocabulary size:', tokenizer.vocab_size)
print('Model is in evaluation mode:', not model.training)
print('\nGenerated sample:')
print(generate_text(
    checkpoint_path,
    prompt='AI engineering',
    max_new_tokens=120,
    temperature=0.8,
    top_k=10,
    device='auto',
))

assert (output_dir / 'best.pt').is_file()
assert (output_dir / 'last.pt').is_file()
assert (output_dir / 'tokenizer.json').is_file()
assert (output_dir / 'metrics.json').is_file()
print('\nSUCCESS: Training, checkpoint files, reload, and generation completed.')


## 4. ????? ????? ???????

???? ??????? ???????? ????? ?? ??????? ????????? ??????? ???????. ??? ??????? ??? ??????? ???????? ???.

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    '/content/model_lab_smoke_artifacts',
    'zip',
    root_dir=output_dir,
)
files.download(archive_path)
